In [0]:
%run ./Config_Parameters

## 🏦 DataBank AI Lab — Module 08: Playgroun & Databricks Apps (Gradio)
**Duration:** ~25 minutes | **Prerequisite:** Module 07 (agent endpoint deployed)

---

### What is a Playground?

**Playground** a no-code chat environment for testing, promotion and compariing large language models (LLMs) and prototyping AI agents. Key Features:
- **Side-by-Side Comparison** add multipole endpoints to compare responses from different models simultaneously.
- **Parameter Tuning** adjust settings like temperature, top K, and maximum tokns to control model behavior and creativity.
- **Tool Integration** connect user-defined functions from the Unity Catalog or Model Context Protocol (MCP) servers to build and test tool-calling agents.
- **Export Options** move your prototypes directly into Databricks Apps or code notebook when you are ready to productionize.


### What You’ll Deploy
```
Browser
  ↓  (HTTPS)
Playground (AI/ML)
  ↓  
Choose Gemma 3 12 B model
  ↓
Tools
  ├── Genie Space        ← Databank-Financial-Advisor
  ├── AI Search          ← Product-Docs-Index  
  └── UC Functions       ← calculate_customer_risk, flag_suspicious_transactions, get_portfolio_summary

```




- AI/ML -> **Playground**
- Select **Gemma 3 12 B** model
- Click **Task** -> **Add** 
  - **Genie Space**
  - **AI Search Index**
  - **UC Functions (calculate_customer_risk, flag_suspicious_transactions, get_portfolio_summary)**
- **System Prompt**

  ```Use Vector Search Index to 1. Search DataBank product brochures, FAQs, and terms & conditions, product knowledge and document questions2. Use for questions about product features, interest rates, fees, eligibility criteria, FSCS protection, or regulatory terms.- Use Genie Spaces to 1. Query customer records, account balances, transaction history, and support tickets using natural language SQL.2. Use for any questions about specific customer account data, transaction queries,  portfolio values, spending patterns, or open support tickets.- Use calculate_customer_risk to Calculate a numeric risk score (0-100) for a DataBank customer. Use when asked about a customer's investment risk tolerance, risk appetite, risk profile calculation, or suitability for investment products.- Use get_portfolio_summary to Retrieve a complete portfolio summary for a customer including all accounts, product types, current balances, and total assets. Use for portfolio reviews, portfolio overviews, financial health checks, or account overviews, account balances,- Use fraud_and_anomaly_check to  Check for suspicious, fraudulent, or anomalous transactions for a customer. Use when asked about fraud, unusual activity, blocked transactions, or security concerns on an account.Always:1. If a customer ID is mentioned, call get_portfolio_summary and calculate_risk_score first.2. For product recommendations, consider the customer’s risk score before recommending.3. For investment products, always include: Capital at risk - investments can fall as well as rise.4. Be concise and professional. Financial advisors are busy.```

![](./img/Playground_1.jpg) 

### Click **Get Code**

![](./img/Playground_2.jpg) 

### Select **Export to Databricks Apps (Recommended)**

Select ML Model that we created as part of **00_setup_prerequisites** notebook and hit **Export**
![](./img/Playground_3.jpg) 

### **Databricks APP gets deployed**

- A Databricks Application will be deployed using https://github.com/databricks/app-templates as a tempalte

### Connect Application with AI Gateway

#### Update agent.py file

- ***add ```use_ai_gateway=True``` while doing ```AsyncDatabricksOpenAI``` function***
- ***Update Model name to your AI Gateway name***

![](./img/Playground_4.jpg)
```
```

### Agent.py - changes

Replace code base in your agent.py in case application is giving error to render chat session.

```

def get_mcp_user_workspace_client():
    # Uncomment the line below to enable on-behalf-of-user authentication
    # return get_user_workspace_client()
    return None


def init_mcp_servers():
    user_workspace_client = get_mcp_user_workspace_client()
    return [
        McpServer(
            name=name,
            url=build_mcp_url(url, user_workspace_client),
            workspace_client=user_workspace_client,
        )
        for (name, url) in MCP_SERVERS
    ]

def create_agent(mcp_servers: List[MCPServer]) -> Agent:
    return Agent(
        name=NAME,
        instructions=SYSTEM_PROMPT,
        model=MODEL,
        mcp_servers=mcp_servers,
    )


def _extract_text(content):
    """Extract plain text from Responses API content (string or list of blocks)."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, dict):
                # output_text (assistant), input_text (user), or generic text
                text = block.get("text", "")
                if text:
                    parts.append(text)
            elif isinstance(block, str):
                parts.append(block)
        return "\n".join(parts)
    return str(content) if content else ""


def normalize_input_items(items):
    """Normalize Responses API input items to chat-completions-style dicts
    that the OpenAI Agents SDK can pass to the model.

    Only user and assistant *message* items are kept; function_call and
    function_call_output items from prior turns are dropped because the
    agent will make its own tool calls. Consecutive same-role messages
    are merged to satisfy models that require strict alternating roles.
    """
    messages = []
    for item in items:
        dumped = item.model_dump()
        item_type = dumped.get("type")
        role = dumped.get("role")

        # Only keep message-type items with user or assistant role.
        # Drop function_call, function_call_output, and anything else.
        if item_type != "message" or role not in ("user", "assistant"):
            continue

        text = _extract_text(dumped.get("content", ""))
        if not text:
            continue

        messages.append({"role": role, "content": text})

    # Merge consecutive same-role messages (safety net)
    merged = []
    for msg in messages:
        if merged and merged[-1]["role"] == msg["role"]:
            merged[-1]["content"] += "\n" + msg["content"]
        else:
            merged.append(msg)

    return merged


@invoke()
async def invoke(request: ResponsesAgentRequest) -> ResponsesAgentResponse:
    mcp_servers = init_mcp_servers()
    async with MCPServerManager(servers = mcp_servers, connect_in_parallel=True) as manager:
        agent = create_agent(manager.active_servers)
        messages = normalize_input_items(request.input)
        result = await Runner.run(agent, messages)
        return ResponsesAgentResponse(output=[item.to_input_item() for item in result.new_items])


@stream()
async def stream(request: dict) -> AsyncGenerator[ResponsesAgentStreamEvent, None]:
    mcp_servers = init_mcp_servers()
    async with MCPServerManager(servers = mcp_servers, connect_in_parallel=True) as manager:
        agent = create_agent(manager.active_servers)
        messages = normalize_input_items(request.input)
        result = Runner.run_streamed(agent, input=messages)

        async for event in process_agent_stream_events(result.stream_events()):
            yield event

```

### Track requests via Experiemnts

#### Click -> ```Experiments``` -> ```<username>_databank_ai_lab experiment```
| | |
|-----|------|
|![](./img/Playground_5.jpg) | ![](./img/Playground_6.jpg) |


## ✅ Module 08 Complete — Checkpoint

| Check | Expected |
|-------|----------|
| Databricks Application Created | Able to query and able to see how Rate Limits, Guardrail works |
| Experiment Tracing | Able to see traces via underlying Experiment |


---

### 🚀 Next: Module 09 — Evaluation
Open **`09_evaluation_llm_judge`** to measure agent response quality using LLM-as-a-judge.